## Classes, Objects, self, `__init__`, and Variables

## why do we even need `object`??? 

~~ object  in an instace of a class....~~

Le't forget `car`, `dog`,  `student` etc.

suppose we're working with a machine learning model. 

In [ ]:
from sklearn.ensemble import RandomForestClassifier 

model = RandomForestClassifier(
    n_estimator = 300, 
    max_depth = 31, 
    random_state = 42
)
model.fit(X_train, y_train)
prediction = model.predict(X_test)

here, what is `model`. It's not just a variable, 
It's an object which represent the state + behavior.

* Model remeber the `configuration`, `behavior`
```
RandomForest model
│
├── Configuration
│   ├── n_estimators = 300
│   ├── max_depth = 12
│   └── random_state = 42
│
├── Learned state
│   ├── estimators_
│   ├── feature_importances_
│   └── ...
│
└── Behavior
    ├── fit()
    ├── predict()
    ├── score()
    └── ...
```

object is an instace of a class.. 

better to say... 

An `object` packages the `state` + `operations` that works on that state. 

OOP give us a natural unit. 
```
             MODEL OBJECT
        ┌────────────────────┐
        │                    │
        │   STATE            │
        │   parameters       │
        │   learned data     │
        │                    │
        │   BEHAVIOR         │
        │   fit()            │
        │   predict()        │
        │   score()          │
        │                    │
        └────────────────────┘
```

## What is a class then ?

- A reusable template, which groups data(`variables`) and behaviors (`functions`) into a single logical package. 

- Class is defination of behavior and structure. 


## Object have their own states:
```
model_a = RandomForestClassifier(n_estimators=100)

model_b = RandomForestClassifier(n_estimators=500)
```
both are coming from same class, but have different states .

```
model_a.fit(X1, y1)
model_b.fit(X2, y2)
```
They also have different learned states. 

## `Self`

In [ ]:
class SomeModel:

    def predict(self, X):
        ...

model.predict(X_test)


## This should automatically be understood as 
SomeModel.predict(model, X_test)
# when we've other model2 object.

SomeModel.predict(model_2, X_test)


Why does `self` exist??
- Because the method (`predict`) need to know which `model's state` it is operating on. 

In [10]:
## example 
class MeanPredictor:
    def fit(self, y):
        self.mean_ = sum(y) / len(y)
        return self
    def predict(self, X):
        return [self.mean_] * len(X)
    

In [ ]:
model = MeanPredictor()
model.fit([20,30,40])
model.predict([1])

## Here predict is accessing self.mean_ = 30

[30.0]

In [15]:
model2 = MeanPredictor()
model2.fit([100, 200, 300])
model2.predict([600])

# Here, we're talking about self.mean_ = 200, basically context is chaned to model2 now. 

[200.0]

In [ ]:
model.fit([20,30,40])
# this is equivalent to below (for understanding purpose)
## basically self is the `model` itself.
MeanPredictor.fit(model, [20, 30, 40])


* Basically python knows which model/model2 to fit/predict through `self` keyword. 
- Infact `self` is not keyword, just a convention followed, you can write `this` or anything. 

## What is `__init__()` then.

- Now that we've an object, we want it to remember its initial states. 


In [ ]:
# so while writing
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12
)
# the newly created object will remember 
n_estimators = 300
max_depth = 12

** this is why `__init__()` is commonly used for. 
* `Initiliziing the states`

In [ ]:
class Model:
    def __init__(self, learning_rate, epochs):
        self.learning_rate = learning_rate
        self.epochs = epochs

model = Model(0.01, 100)

* Thus the Model object now becomes:- 
```
model
│
├── learning_rate → 0.01
├── epochs → 100
│
├── fit()
└── predict()
```

### why does  `__init__()` has `self`??
- so writing `model = Model(0.01)` is equivalent to understanding 
`Model.__init__(model, 0.01)`, 
- So, everything will be behaved as per the states of `model`. 
- That's what `self` is referring to `model` here.

## Instance Variables v/s Class Variables. 

In [ ]:
class Experiment:

    framework = "Python"

    def __init__(self, name):
        self.name = name

In [ ]:
xyz = Experiment('abc')
pqr  = Experiment('random')

* Framework is `class variable`, name is `instance variable` referring only to that instance of any class object. 
* 'abc` and 'random' are instance variables limited only for the scope of their respective object. 


### Why use class variables?
- when the value is shared by all the instances. 
- eg - `timeout = 20`

To remember :- 

`Class` - Shared defination + shared behavior   
`Object` - 1 particular instance + its own state  
`self` - Which particular instance we're talking about.  

In [16]:
class Model:

    framework = "sklearn"

    def __init__(self, name):
        self.name = name


m = Model("Fraud Detector")

print(m.__dict__)
print(Model.__dict__.keys())

{'name': 'Fraud Detector'}
dict_keys(['__module__', 'framework', '__init__', '__dict__', '__weakref__', '__doc__'])


* we can see that the `name` belong to the object instance, but the `framework` is defined on the class. 

In [18]:
class Config:
    timeout = 40

a = Config()
b = Config()
a.timeout = 99
print(a.timeout)
print(b.timeout)

print(Config.timeout)

99
40
40


* this is because, `attribute lookup` is first checked up in the instance .
* LEGB, local, Enclosing, Global, concept.. 

# Interview Focused qns:- 

Explain `self`
- it refers to the current object...
- better explain with example. 
- `model.predict(x)`, here, python passes model as the first argument to the method. allowing the method(predict), to access that instance's state through `self`. 

### Mental model
```
             CLASS
              │
      ┌───────┴────────┐
      │                │
   methods         class attrs
      │                │
      └───────┬────────┘
              │
       ┌──────┴──────┐
       ▼             ▼
    object A      object B
       │             │
   own state      own state
       │             │
      self          self
```

# Instance, Class & Static methods + @Property

This section will help us understand, `who should this method belong to`. 
 One object, whole class or neither?

- the main difference b/w them will be  `what data they access to and how are they called.`

## 1 Instance Method. 

* work on this object. 

- are bound to a specific object instance and can modify its unique state only 

```
1. model_a.predict(X)
2. model_b.predict(X)
```
* Here, same method `predict` is being used, but each call operate on a different model.

* We're telling `predict` to work on the respective object model. 


* _ So use `instance method ` when the operation needs a particular object's state. 

 # !
 Why do we need them, 

And when do we need them all. 

why not one only for everything..... 

## 2 Class Method
## 3 Static Method

### why we need  them all....
* Imagine we're building a Bank Account System. And we need these different method working as team to handle different responsibility. 


In [ ]:
class BankAccount:
    interest_rate  = 0.03 # shared by all accounts

    def __init__(self, owner, balance):
        self.owner = owner # unique to this account
        self.balance = balance # unique to this account

    # *---- Instance Method --- 
    # Need: Modify this person's money only. 
    def withdraw(self, amount):
        if amount <= self.balance:
            self.balance -= amount
            print(f"Withdrew ${amount}. New balance: ${self.balance}")

    

    # * ----- Class Method ---- 
    # Need: A factory to create a specific type of account with standard defaults. 
    @classmethod
    def create_child_account(cls, owner):
         # starting a child account with class blueprint (cls)
         # ! owner = owner, balance = 10
         # this `cls` is same as `self` for specific object. 
         return cls(owner, balance = 10)

    # * --- Static Method --- 
    def is_valid_currency(currency_code):
        # Purity utility check - true/False
        return currency_code in ["USD", "EUR", "GBP"]
    

* static Method - `BankAccount.is_valid_currency('USD')`. 
This checks the currency validity, even before we start. 

* Class method - `kids_acc = BankAccount.create_child_account("Leo")`
Builds a specilized kids' account

* Instance Method - `kids_acc.withdraw(6)`, spends money from that account .

__ Other examples ___

In [ ]:
class Employee:
    # Class Attribute: Base standard working hours for everyone
    standard_work_hours = 40 

    def __init__(self, name, hourly_rate):
        # Instance Attributes: Unique to each employee 
        self.name = name
        self.hourly_rate = hourly_rate

    # *--- 1. INSTANCE METHOD ---
    # Need: Accesses unique data (hourly_rate) to calculate one specific person's pay.
    def calculate_weekly_pay(self):
        return self.hourly_rate * self.standard_work_hours

    # *--- 2. CLASS METHOD ---
    # Need: A factory to build a preset object when you only have a raw text string.
    @classmethod
    def from_csv_string(cls, csv_text):
        # Parses a string like "Alice,25" into separate variables
        name, rate_str = csv_text.split(",")
        # Creates and returns the new Employee object using 'cls'
        return cls(name, float(rate_str))

    # *--- 3. STATIC METHOD ---
    # Need: A simple office rule check. It doesn't need to know who the employee is.
    # This is blind to everything the object has. ...(no object attribute information.)
    @staticmethod
    def is_work_day(day_name):
        # Pure logic: returns True for weekdays, False for weekends
        return day_name.lower() not in ["saturday", "sunday"]


### How you use them together. 

In [21]:
 # * 1. Static method. 
# No employee object exists yet. 
if Employee.is_work_day('Monday'):
    print("office open")

 # * 2. Class method. 
# To import a new hire into the system. 
# this create an object for us. 
new_hire = Employee.from_csv_string("alan, 33")


office open


In [23]:
new_hire.__dict__

{'name': 'alan', 'hourly_rate': 33.0}

* We can see thta this `new hire` has got the attributes of the Employee Class. 
* We created a new object, and `cls` tagged it to the the class attributes. 

* Now that the object is created, we can use the instance method, find his pay.

In [24]:
alan_pay = new_hire.calculate_weekly_pay()
print(f"{new_hire.name} earned ${alan_pay}") 

alan earned $1320.0


## Why one method cannot do it all by itself?? 

* `is_work_day`: can't be an instance method, because HR just need to check if the day is workday, `before any employees are loaded` into the system. 

* `from_csv_string` can't be a normal instance method, because you can't call an instance method on object which `hasn't been created yet`. 

* `calculate_weekly_pay` can't be static method, becoz a static method is `blind to alan's hourly rate`. 

## Qns:

1. Static method is faster. ?
- Not major performance advantage, rather use it when the operation logically belongs to the class, but doesn't need instance or class state. 


## @Property

* this decorator turns a `method` into a `read only variable (attribute)`

* Allows to call method without `()`